# Host response: does a mouse cell answer like a human cell?


**The open point this closes (instruction 71, product 3).** The host response atlas held human and mouse signatures apart, because they share no gene identifiers. Mapped onto the gene symbol they share, they can be compared, and the question becomes answerable: does the host response to the same parasite perturbation look the same in a mouse cell as in a human cell?

**The mapping, and its limit.** Symbols only -- mouse `Bfd1` meets human `BFD1`. A symbol that names more than one gene in a species is dropped rather than guessed, so a gene family where the species differ in member count is simply absent. This is not tested orthology and is not offered as such.

Built 2026-11-08.

In [1]:
import json, os, numpy as np, pandas as pd
from parascape import perturbation_atlas as PA
store = PA.host_store()
crossed = PA.cross_species(store)
reg = crossed['registry']
human = [c for c in crossed['lfc'].columns if reg.loc[c, 'species'] == 'human']
mouse = [c for c in crossed['lfc'].columns if reg.loc[c, 'species'] == 'mouse']
both = int((crossed['lfc'][human].notna().any(axis=1) & crossed['lfc'][mouse].notna().any(axis=1)).sum())
print(len(human), 'human and', len(mouse), 'mouse signatures over', crossed['lfc'].shape[0], 'symbols')
print(both, 'symbols measured in both species;', crossed['dropped']['rows_without_symbol'],
      'gene-signature values dropped for want of a one-to-one symbol')
pd.DataFrame([crossed['dropped']])

99 human and 6 mouse signatures over 31151 symbols
10881 symbols measured in both species; 3123729 gene-signature values dropped for want of a one-to-one symbol


,rows_without_symbol,symbols
0,3123729,31151


## 1 · The one perturbation both species state

The treated label is normalised the way `replicate_groups` normalises it, then grouped across species. A perturbation named by only one species cannot be crossed and is listed.

In [2]:
norm = reg.treated.astype(str).str.lower().str.replace(r'[^a-z0-9]+', '', regex=True)
reg = reg.assign(norm=norm)
def family(text):
    for key in ('bfd1', 'bfd2', 'gra18', 'myr1', 'lps', 'ifng'):
        if key in text:
            return key
    return ''
reg['family'] = reg.norm.map(family)
shared = sorted({f for f in reg.loc[mouse, 'family'] if f} & {f for f in reg.loc[human, 'family'] if f})
print('perturbations stated in both species:', shared)
print('mouse only:', sorted({f for f in reg.loc[mouse, 'family'] if f} - set(shared)))
reg.loc[reg.family != '', ['study', 'species', 'treated', 'family']]

perturbations stated in both species: ['bfd1']
mouse only: ['bfd2', 'gra18', 'lps']


,study,species,treated,family
column,,,,
host_lfc_GSE103113_0,GSE103113,mouse,gra18,gra18
host_lfc_GSE103113_1,GSE103113,mouse,gra18,gra18
host_lfc_GSE103113_2,GSE103113,mouse,lps,lps
host_lfc_GSE103113_3,GSE103113,mouse,lps,lps
host_lfc_GSE125120_6,GSE125120,human,myr1 ko,myr1
host_lfc_GSE125120_7,GSE125120,human,myr1 ko,myr1
host_lfc_GSE132238_14,GSE132238,human,{delta}bfd1,bfd1
host_lfc_GSE132238_15,GSE132238,human,{delta}bfd1,bfd1
host_lfc_GSE132238_17,GSE132238,human,{delta}bfd1,bfd1


## 2 · Does the mouse signature find the human one?

Each mouse signature is correlated (Spearman, on the symbols both measure) with every human signature. The rank of the best human signature **of the same perturbation** is the answer; chance is the rank an unrelated signature would get, which is half the number of human signatures.

In [3]:
MIN_SHARED = 500
lfc = crossed['lfc']
rows = []
for m in mouse:
    fam = reg.loc[m, 'family']
    scores = {}
    for h in human:
        pair = lfc[[m, h]].dropna()
        if len(pair) < MIN_SHARED:
            continue
        scores[h] = pair.corr(method='spearman').iloc[0, 1]
    if not scores:
        rows.append(dict(mouse_signature=m, perturbation=fam, comparable=0, verdict='no human signature shares enough symbols'))
        continue
    ranked = pd.Series(scores).sort_values(ascending=False)
    same = [h for h in ranked.index if reg.loc[h, 'family'] == fam and fam]
    best = int(min(ranked.index.get_loc(h) for h in same) + 1) if same else None
    rows.append(dict(mouse_signature=m, perturbation=fam or '(not one of the shared kinds)',
                     comparable=len(ranked), same_perturbation_signatures=len(same),
                     best_same_rank=best, chance_rank=round((len(ranked) + 1) / 2, 1),
                     best_rho=round(float(ranked.iloc[0]), 3),
                     best_same_rho=round(float(ranked[same].max()), 3) if same else None,
                     top_match=ranked.index[0], top_match_label=reg.loc[ranked.index[0], 'contrast_label']))
X = pd.DataFrame(rows)
X.to_csv(os.path.join(OUT, 'cross_species_recovery.tsv'), sep='\t', index=False)
X

,mouse_signature,perturbation,comparable,same_perturbation_signatures,best_same_rank,chance_rank,best_rho,best_same_rho,top_match,top_match_label
0,host_lfc_GSE103113_0,gra18,99,0,NaN,50,0.133,NaN,host_lfc_GSE223819_153,treatment: 50um iaa vs vehicle (pbs) (genotype=me49/tir1)
1,host_lfc_GSE103113_1,gra18,99,0,NaN,50,0.142,NaN,host_lfc_GSE125120_4,condition: infected vs bystander (genotype=wild type)
2,host_lfc_GSE103113_2,lps,99,0,NaN,50,0.181,NaN,host_lfc_GSE193614_60,genotype: unstated vs wild type (reference background)
3,host_lfc_GSE103113_3,lps,99,0,NaN,50,0.138,NaN,host_lfc_GSE125120_7,genotype: myr1 ko vs wild type (condition=infected)
4,host_lfc_GSE223621_139,bfd1,99,14,1,50,0.203,0.203,host_lfc_GSE223877_166,"genotype: me49delta ku80delta bfd1::bfd1-tydelta hxgprt::ptub1-dd-bfd1-tydelta bfd2::tdtomato vs me49delta ku80delta bfd1::bfd1-tydelta hxgprt::ptub1-dd-bfd1-ty (treatment=standard medium, +3um shield-1)"
5,host_lfc_GSE223621_140,bfd2,99,0,NaN,50,0.131,NaN,host_lfc_GSE223869_162,"genotype: me49delta ku80delta bfd1::bfd1-tydelta bfd2 vs me49delta ku80delta bfd1::bfd1-ty (treatment=alkaline stress media, ambient co2)"


## 3 · What it says

A mouse signature whose same-perturbation human partner ranks near the top means the host response crosses the species; one that ranks at chance means the two cells answer differently, or that the symbol mapping loses what matters. Both readings are stated, never only the hopeful one.

In [4]:
tested = X[X.best_same_rank.notna()]
summary = dict(mouse_signatures=len(X), testable=len(tested),
               median_best_same_rank=(float(tested.best_same_rank.median()) if len(tested) else None),
               median_chance_rank=(float(tested.chance_rank.median()) if len(tested) else None),
               symbols_in_both=both)
json.dump(summary, open(os.path.join(OUT, 'summary.json'), 'w'), indent=1)
print(summary)
tested[['mouse_signature', 'perturbation', 'best_same_rank', 'chance_rank', 'best_same_rho', 'top_match_label']]

{'mouse_signatures': 6, 'testable': 1, 'median_best_same_rank': 1.0, 'median_chance_rank': 50.0, 'symbols_in_both': 10881}


,mouse_signature,perturbation,best_same_rank,chance_rank,best_same_rho,top_match_label
4,host_lfc_GSE223621_139,bfd1,1,50,0.203,"genotype: me49delta ku80delta bfd1::bfd1-tydelta hxgprt::ptub1-dd-bfd1-tydelta bfd2::tdtomato vs me49delta ku80delta bfd1::bfd1-tydelta hxgprt::ptub1-dd-bfd1-ty (treatment=standard medium, +3um shield-1)"
